# Python `str.translate()`: Master Fast Character Mapping & Deletions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/plus2net/Python-basics/blob/main/python_string_translate_practice.ipynb)

Welcome to the interactive companion notebook for the **[plus2net Python str.translate() Tutorial](https://www.plus2net.com/python/string-translate.php)**.

### Learning Objectives:
1. Build translation tables using all 3 forms of `str.maketrans()`.
2. Perform simultaneous, collision-free character swapping in a single pass.
3. Mass-delete characters (such as punctuation) with native C speed.
4. Implement DNA reverse complements and ROT13 ciphers.
5. Validate implementations with automated self-checking unit assertions.

## 1. The Three Signatures of `str.maketrans()`

- **1 Argument (dict)**: `{char: replacement}`
- **2 Arguments (strings)**: `maketrans(x, y)` maps characters by position
- **3 Arguments (x, y, z)**: maps `x` to `y`, deletes characters in `z`

In [ ]:
# Form 1: Dict
t1 = str.maketrans({"<": "&lt;", ">": "&gt;", "@": None})
assert "<user@host>".translate(t1) == "&lt;userhost&gt;"

# Form 2: Two equal-length strings
t2 = str.maketrans("abc", "xyz")
assert "abcdef".translate(t2) == "xyzdef"

# Form 3: Two strings + deletion set
t3 = str.maketrans("12", "AB", "- ")
assert "1-2-1 2".translate(t3) == "ABAB"

print("All 3 maketrans signatures verified!")


## 2. Collision-Free Simultaneous Swapping vs. `replace()` Chaining

Chained `.replace("A", "B").replace("B", "A")` collides and ruins the swap.
`str.translate()` swaps characters simultaneously without collisions.

In [ ]:
source = "ABBA"

# Chained replace fails
flawed = source.replace("A", "B").replace("B", "A")
print("Chained replace: ", flawed)  # 'AAAA'
assert flawed == "AAAA"

# Simultaneous translate succeeds
swap_table = str.maketrans("AB", "BA")
correct = source.translate(swap_table)
print("translate() swap:", correct) # 'BAAB'
assert correct == "BAAB"

print("\nSimultaneous swapping assertion passed!")


## 3. High-Speed Character Deletion

Using `str.maketrans("", "", delete_set)` deletes characters in pure C.

In [ ]:
import string

doc = "Hello, world! Welcome to Python: fast, clean, & robust."
clean_doc = doc.translate(str.maketrans("", "", string.punctuation))

print("Punctuation stripped:", clean_doc)
assert "!" not in clean_doc and "," not in clean_doc

print("Character deletion assertion passed!")


## 4. Real-World Engineering Recipes

Explore three common software engineering patterns utilizing `str.translate()`:
1. DNA Reverse Complement Generator
2. ROT13 Cipher Implementation
3. Fast Phone Number Digit Extractor

In [ ]:
# Recipe 1: DNA Reverse Complement
def dna_rev_comp(seq: str) -> str:
    comp_t = str.maketrans("ATCGatcg", "TAGCtagc")
    return seq.translate(comp_t)[::-1]

dna = "ATGCGATC"
rc = dna_rev_comp(dna)
print("DNA:", dna, "-> RevComp:", rc)
assert rc == "GATCGCAT"

# Recipe 2: ROT13 cipher
def rot13_cipher(text: str) -> str:
    l_in = string.ascii_lowercase
    l_out = l_in[13:] + l_in[:13]
    u_in = string.ascii_uppercase
    u_out = u_in[13:] + u_in[:13]
    t = str.maketrans(l_in + u_in, l_out + u_out)
    return text.translate(t)

enc = rot13_cipher("Hello World")
print("ROT13:", enc)
assert rot13_cipher(enc) == "Hello World"

# Recipe 3: Fast digit extraction
def get_digits(phone_str: str) -> str:
    non_d = "".join(chr(i) for i in range(128) if not chr(i).isdigit())
    return phone_str.translate(str.maketrans("", "", non_d))

assert get_digits("+1 (800) 555-0199 ext. 42") == "1800555019942"

print("\nAll engineering recipes passed!")


## 5. Production Implementation & Self-Checking Unit Assertions

This production-grade `CharacterTranslator` class encapsulates simultaneous swapping, punctuation deletion, and DNA complements.

In [ ]:
class CharacterTranslator:
    @staticmethod
    def swap_tokens(text: str, set_a: str, set_b: str) -> str:
        """Simultaneously swaps characters in set_a with set_b."""
        table = str.maketrans(set_a + set_b, set_b + set_a)
        return text.translate(table)

    @staticmethod
    def strip_punctuation(text: str) -> str:
        """Removes all standard ASCII punctuation in a single C pass."""
        table = str.maketrans("", "", string.punctuation)
        return text.translate(table)

    @staticmethod
    def dna_complement(dna_seq: str) -> str:
        """Calculates reverse complement of DNA sequence."""
        table = str.maketrans("ATCG", "TAGC")
        return dna_seq.upper().translate(table)[::-1]


# ==========================================
# UNIT ASSERTION SUITE
# ==========================================
translator = CharacterTranslator()

# 1. Simultaneous swapping without collision
assert translator.swap_tokens("ABBA", "A", "B") == "BAAB"
assert translator.swap_tokens("XYZ", "X", "Z") == "ZYX"

# 2. Fast punctuation deletion
raw = "Hello, world! How are you?"
assert translator.strip_punctuation(raw) == "Hello world How are you"

# 3. DNA reverse complement
assert translator.dna_complement("ATGCGATC") == "GATCGCAT"
assert translator.dna_complement("AAAA") == "TTTT"

# 4. 1-arg dictionary form with multi-char replacement value
html_table = str.maketrans({"<": "&lt;", ">": "&gt;"})
assert "<div>".translate(html_table) == "&lt;div&gt;"

# 5. ValueError on unequal length strings in maketrans
try:
    str.maketrans("abc", "xy")
    assert False, "Expected ValueError not raised"
except ValueError as e:
    assert "equal length" in str(e)

print("All str.translate() production unit assertions passed successfully!")
